# 08.02 -- Reward Models: Learning What Humans Prefer

**Module:** 08 -- Alignment  
**Notebook:** 2 of 5  
**Prerequisites:** 08.01 (What Is Alignment)

---

## Learning Objectives

1. Understand the architecture of a reward model and how it differs from a language model
2. Implement the Bradley-Terry preference model from first principles
3. Train a reward model on pairwise preference data using the standard loss
4. Identify and measure common reward model failure modes: sycophancy bias, length bias, distribution shift
5. Evaluate reward model quality with preference accuracy and calibration metrics

---

## 1. What Is a Reward Model?

In the RLHF pipeline, the reward model (RM) is a neural network trained to predict which of two responses a human would prefer. It takes a prompt and a response as input and outputs a scalar score representing quality.

Architecturally, a reward model is a standard language model with its language modelling head removed and replaced by a single linear layer that maps the final hidden state to a scalar:

```
Standard LM:    [token embeddings] -> [transformer layers] -> [LM head, vocab_size] -> next token probs

Reward Model:   [token embeddings] -> [transformer layers] -> [reward head, 1]      -> scalar score
```

The reward model is initialised from the SFT model (not the base pretrained model) because it needs to understand the instruction-following format.

---

## 2. The Bradley-Terry Model: Mathematical Foundation

The reward model is trained using the **Bradley-Terry model** from statistics, which describes the probability that item A is preferred over item B given their underlying quality scores:

```
P(A > B) = sigmoid(r(A) - r(B))
         = exp(r(A)) / (exp(r(A)) + exp(r(B)))
```

where r(x) is the reward model's scalar output for response x.

The training loss maximises the log probability that the chosen response scores higher than the rejected response:

```
Loss = -E[log sigmoid(r(chosen) - r(rejected))]
```

This is a binary cross-entropy loss applied to the difference of scores. No explicit labels are needed beyond the ordering (chosen > rejected).

In [ ]:
# Environment setup.
#
# All libraries used in this notebook. The key addition over previous
# notebooks is scipy.stats, which we use for calibration analysis,
# and sklearn.metrics for standard classification metrics applied
# to the reward model's pairwise preference accuracy.

!pip install transformers peft datasets trl accelerate matplotlib scipy scikit-learn -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

## 3. Implementing the Bradley-Terry Loss From Scratch

Before using the HuggingFace implementation, we build the loss from first principles to ensure the mathematics is clear.

In [ ]:
# Bradley-Terry preference loss, implemented from the mathematical definition.
#
# The function takes two batches of reward scores -- one for chosen responses
# and one for rejected -- and returns the scalar loss.
#
# The key formula is:
#   loss = -mean(log(sigmoid(r_chosen - r_rejected)))
#
# Numerically, log(sigmoid(x)) = -log(1 + exp(-x)) = -softplus(-x).
# Using F.logsigmoid instead of log(sigmoid(...)) avoids numerical underflow
# when the argument is a large negative number.
#
# The margin parameter adds a minimum required gap between chosen and rejected
# scores. Without a margin, the loss approaches zero whenever r_chosen > r_rejected
# by any amount. With a margin of m, we require r_chosen > r_rejected + m,
# which prevents overconfident but poorly-separated scores.

def bradley_terry_loss(
    r_chosen:   torch.Tensor,   # (batch,) scalar reward for chosen responses
    r_rejected: torch.Tensor,   # (batch,) scalar reward for rejected responses
    margin:     float = 0.0,    # minimum required score gap
) -> torch.Tensor:
    """
    Standard reward model training loss from the RLHF literature.

    Returns the scalar mean loss over the batch.
    """
    # The difference is what matters -- absolute scale is arbitrary
    diff = r_chosen - r_rejected - margin
    # log(sigmoid(diff)) is the log-probability that chosen is preferred
    return -F.logsigmoid(diff).mean()


def preference_accuracy(
    r_chosen:   torch.Tensor,
    r_rejected: torch.Tensor,
) -> float:
    """
    Fraction of pairs where the reward model correctly ranks chosen > rejected.

    This is the primary evaluation metric for reward models. A random
    baseline achieves 50%; a good reward model should exceed 70%.
    State-of-the-art models achieve 75-85% on held-out human preference data.
    """
    return (r_chosen > r_rejected).float().mean().item()


# Demonstrate the loss behaviour under three scenarios:
# (a) correct ordering with large margin -- loss should be near zero
# (b) correct ordering with tiny margin -- loss is positive but small
# (c) wrong ordering -- loss is large, gradient pushes scores apart

scenarios = [
    ("Strongly correct (chosen >> rejected)",
     torch.tensor([2.5, 3.0, 2.8]),
     torch.tensor([0.1, 0.5, 0.2])),
    ("Weakly correct (chosen > rejected by 0.1)",
     torch.tensor([1.1, 1.1, 1.1]),
     torch.tensor([1.0, 1.0, 1.0])),
    ("Correct on average, one wrong pair",
     torch.tensor([2.0, 0.5, 2.5]),
     torch.tensor([0.5, 1.5, 0.5])),
    ("All wrong (chosen << rejected)",
     torch.tensor([0.1, 0.2, 0.3]),
     torch.tensor([2.0, 2.5, 2.2])),
]

print(f"{'Scenario':<50} {'Loss':>8} {'Accuracy':>10}")
print("-" * 72)
for label, r_c, r_r in scenarios:
    loss = bradley_terry_loss(r_c, r_r)
    acc  = preference_accuracy(r_c, r_r)
    print(f"{label:<50} {loss.item():>8.4f} {acc:>10.1%}")

In [ ]:
# Visualise the Bradley-Terry loss landscape.
#
# This plot shows how the training loss depends on the score difference
# (r_chosen - r_rejected). Understanding this curve is essential for
# diagnosing reward model training problems.
#
# The curve has three important regions:
#   - diff << 0: high loss, large gradient (model is badly wrong)
#   - diff near 0: moderate loss, meaningful gradient (uncertain region)
#   - diff >> 0: near-zero loss, saturated gradient (model is confident)
#
# Saturation in the right tail means the model does not learn from very
# easy examples. This is why hard example mining is important in practice.

diffs = np.linspace(-5, 5, 300)
loss_values = -np.log(1 / (1 + np.exp(-diffs)))   # = softplus(-diff)
grad_values = 1 / (1 + np.exp(diffs))              # sigmoid(-diff) = gradient magnitude

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(diffs, loss_values, color='#1f77b4', linewidth=2.5)
ax.axvline(0, color='gray', linestyle='--', alpha=0.6, label='Decision boundary')
ax.fill_between(diffs, 0, loss_values,
                where=(diffs < 0), alpha=0.15, color='#d62728', label='Wrong ordering')
ax.fill_between(diffs, 0, loss_values,
                where=(diffs >= 0), alpha=0.15, color='#2ca02c', label='Correct ordering')
ax.set_xlabel('r_chosen - r_rejected', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('Bradley-Terry Loss vs Score Difference', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_ylim(0, 4)

ax = axes[1]
ax.plot(diffs, grad_values, color='#ff7f0e', linewidth=2.5)
ax.axvline(0, color='gray', linestyle='--', alpha=0.6)
ax.set_xlabel('r_chosen - r_rejected', fontsize=11)
ax.set_ylabel('Gradient Magnitude', fontsize=11)
ax.set_title('Gradient Magnitude vs Score Difference\n'
             '(Saturates for easy examples -- motivates hard example mining)', fontsize=12)
ax.grid(alpha=0.3)
ax.annotate('Max gradient at\ndecision boundary',
            xy=(0, 0.5), xytext=(1.5, 0.6),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9)
ax.annotate('Near-zero gradient\n(saturated, easy pairs)',
            xy=(4, grad_values[np.searchsorted(diffs, 4)]),
            xytext=(2.5, 0.25),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9)

plt.suptitle('Bradley-Terry Loss Characteristics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/08_bt_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Reward Model Architecture

A reward model is a language model with a scalar prediction head. We implement this architecture from scratch to understand every component before moving to the HuggingFace implementation.

In [ ]:
# Reward model architecture built on top of a pretrained language model.
#
# The design follows the InstructGPT paper closely:
#   1. Start from the SFT checkpoint (not the base model)
#   2. Remove the unembedding (LM head) layer
#   3. Add a single linear layer: hidden_size -> 1
#   4. The output at the last non-padding token position is the reward
#
# Why the last token? For causal (left-to-right) language models, the
# hidden state at the final token has attended to the entire sequence.
# It therefore encodes a full summary of the prompt and response.
# For encoder models like BERT, the [CLS] token is used instead.
#
# The reward head is initialised with small weights. Large initial rewards
# would produce large gradients on the first batch and destabilise training.

from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel


class RewardModel(nn.Module):
    """
    Reward model for RLHF alignment.

    Wraps a causal language model and adds a scalar prediction head.
    The reward is computed from the hidden state at the last non-padding
    token in the sequence.
    """

    def __init__(self, base_model_name: str, dropout: float = 0.1):
        super().__init__()

        # Load the pretrained model. We use AutoModel (not AutoModelForCausalLM)
        # because we do not need the language modelling head.
        # In practice, start from the SFT checkpoint, not the base model.
        base = AutoModelForCausalLM.from_pretrained(base_model_name)

        # Extract the transformer backbone only -- discard the LM head
        self.transformer = base.transformer  # GPT-2 specific; use .model for LLaMA
        hidden_size = base.config.n_embd     # 768 for GPT-2 small

        # Scalar reward head. Bias=False is common in reward models;
        # the absolute scale of rewards is arbitrary (only differences matter).
        self.reward_head = nn.Linear(hidden_size, 1, bias=False)

        # Optional dropout for regularisation during training
        self.dropout = nn.Dropout(dropout)

        # Small initialisation prevents large initial rewards
        nn.init.normal_(self.reward_head.weight, std=0.02)

    def forward(
        self,
        input_ids:      torch.Tensor,    # (batch, seq_len)
        attention_mask: torch.Tensor,    # (batch, seq_len), 1=real token, 0=pad
    ) -> torch.Tensor:                   # (batch,) scalar rewards

        # Run the transformer backbone
        outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        hidden_states = outputs.last_hidden_state  # (batch, seq_len, hidden_size)

        # Extract the hidden state at the last real (non-padding) token.
        # attention_mask.sum(dim=1) - 1 gives the index of the last real token
        # for each example in the batch, accounting for variable-length sequences.
        last_token_idx = attention_mask.sum(dim=1) - 1  # (batch,)
        batch_idx      = torch.arange(hidden_states.size(0), device=hidden_states.device)
        last_hidden    = hidden_states[batch_idx, last_token_idx]  # (batch, hidden_size)

        # Project to scalar reward
        reward = self.reward_head(self.dropout(last_hidden)).squeeze(-1)  # (batch,)
        return reward

    def get_reward(self, input_ids, attention_mask) -> torch.Tensor:
        """Convenience wrapper that detaches and returns numpy scalar for inference."""
        with torch.no_grad():
            return self(input_ids, attention_mask)


# Initialise the reward model from GPT-2 for demonstration.
# In production, use the SFT checkpoint of your target model.
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

reward_model = RewardModel('gpt2').to(device)

total_params    = sum(p.numel() for p in reward_model.parameters())
trainable       = sum(p.numel() for p in reward_model.parameters() if p.requires_grad)
head_params     = sum(p.numel() for p in reward_model.reward_head.parameters())

print("Reward model parameter breakdown:")
print(f"  Transformer backbone: {total_params - head_params:>12,}  (pretrained, fine-tuned during RM training)")
print(f"  Reward head:          {head_params:>12,}  (trained from scratch)")
print(f"  Total:                {total_params:>12,}")
print()
print("Note: the reward head adds negligible parameters.")
print("Almost all capacity is in the transformer backbone.")

In [ ]:
# Verify the reward model's forward pass produces the correct output shape.
#
# We tokenise two short texts and confirm that the model outputs a single
# scalar per sequence. This is the sanity check to run before any training.
# If the shape is wrong, the Bradley-Terry loss will silently broadcast
# incorrectly and produce nonsense gradients.

test_texts = [
    "### Instruction:\nExplain gradient descent.\n\n### Response:\nGradient descent minimises a loss by iteratively following the gradient downhill.",
    "### Instruction:\nExplain gradient descent.\n\n### Response:\nIt is a thing that makes models learn.",
]

test_batch = tokenizer(
    test_texts,
    return_tensors='pt',
    padding=True,
    truncation=True,
    max_length=128,
).to(device)

with torch.no_grad():
    scores = reward_model(
        test_batch['input_ids'],
        test_batch['attention_mask']
    )

print(f"Input shape:  {test_batch['input_ids'].shape}  (batch=2, seq_len varies)")
print(f"Output shape: {scores.shape}  (one scalar per sequence -- correct)")
print(f"Scores (before training, effectively random): {scores.tolist()}")
print()
print("Before training the reward head, scores are random.")
print("After training on preference data, the first response should score higher.")

## 5. Training the Reward Model

In [ ]:
# Preference dataset for reward model training.
#
# Each example consists of a prompt, a chosen response (preferred by annotators),
# and a rejected response. The categories capture different quality dimensions:
# factual accuracy, code quality, safety, and response format.
#
# In a production setting this dataset would have 10,000-100,000 examples
# collected from human annotators via a labelling interface. Here we use a
# small handcrafted set purely to demonstrate the training loop.

@dataclass
class PreferencePair:
    prompt:   str
    chosen:   str
    rejected: str
    category: str = 'general'


PREFERENCE_DATA = [
    PreferencePair(
        prompt="Explain the vanishing gradient problem.",
        chosen="The vanishing gradient problem occurs when gradients become exponentially small during backpropagation through many layers. Sigmoid and tanh activations have derivatives less than 1, so multiplying them through deep networks causes gradients near the input to approach zero, preventing early layers from learning. Solutions include ReLU activations, residual connections, and batch normalisation.",
        rejected="Gradients vanish sometimes. This makes training hard. You should use better activation functions.",
        category='explanation',
    ),
    PreferencePair(
        prompt="What is the difference between precision and recall?",
        chosen="Precision is the fraction of positive predictions that are actually positive: TP / (TP + FP). Recall is the fraction of actual positives that were correctly predicted: TP / (TP + FN). High precision means few false alarms; high recall means few missed cases. The F1 score is their harmonic mean. Which matters more depends on the application: spam filtering prioritises precision (do not mark legitimate email as spam); cancer screening prioritises recall (do not miss positive cases).",
        rejected="Precision and recall are both metrics. Precision is about accuracy and recall is about coverage. They are used in machine learning.",
        category='explanation',
    ),
    PreferencePair(
        prompt="Write a function to compute the Fibonacci sequence up to n.",
        chosen="def fibonacci(n: int) -> list[int]:\n    \"\"\"Return Fibonacci numbers up to n (inclusive).\"\"\"\n    if n < 0:\n        raise ValueError(f'n must be non-negative, got {n}')\n    sequence = []\n    a, b = 0, 1\n    while a <= n:\n        sequence.append(a)\n        a, b = b, a + b\n    return sequence",
        rejected="def fib(n):\n    result = []\n    for i in range(n):\n        if i == 0: result.append(0)\n        elif i == 1: result.append(1)\n        else: result.append(result[-1] + result[-2])\n    return result",
        category='code',
    ),
    PreferencePair(
        prompt="How should I handle sensitive personal data in a web application?",
        chosen="Handle sensitive data with: (1) Encryption at rest using AES-256; (2) TLS for all data in transit; (3) Hashing passwords with bcrypt or Argon2, never plain SHA; (4) Minimal data collection -- only store what is genuinely needed; (5) Access controls so only authorised code can read sensitive fields; (6) Audit logging for all access. Also review applicable regulations such as GDPR or HIPAA for your jurisdiction.",
        rejected="You should be careful with personal data. Use encryption and follow best practices. Make sure to comply with laws.",
        category='safety',
    ),
    PreferencePair(
        prompt="What causes a model to overfit?",
        chosen="Overfitting occurs when a model learns patterns specific to the training set that do not generalise. Primary causes: (a) insufficient training data relative to model capacity, (b) training for too many epochs without early stopping, (c) no regularisation (dropout, weight decay, data augmentation). Regularisation methods work by adding constraints that prevent the model from memorising individual examples.",
        rejected="Overfitting is when training accuracy is high but test accuracy is low. You can fix it with regularisation and more data.",
        category='explanation',
    ),
] * 8  # Replicate for a larger demo training set


class PreferenceDataset(torch.utils.data.Dataset):
    """
    PyTorch Dataset for pairwise preference data.

    Each example contains tokenised versions of both the chosen and rejected
    response. We tokenise the full (prompt + response) string because the
    reward model must condition on the prompt to produce a meaningful score.
    """

    def __init__(
        self,
        pairs:      List[PreferencePair],
        tokenizer,
        max_length: int = 256,
    ):
        self.pairs      = pairs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.pairs)

    def _tokenize(self, prompt: str, response: str) -> Dict:
        text = f"### Instruction:\n{prompt}\n\n### Response:\n{response}"
        return self.tokenizer(
            text,
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt',
        )

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        pair = self.pairs[idx]
        chosen   = self._tokenize(pair.prompt, pair.chosen)
        rejected = self._tokenize(pair.prompt, pair.rejected)
        return {
            'chosen_input_ids':        chosen['input_ids'].squeeze(0),
            'chosen_attention_mask':   chosen['attention_mask'].squeeze(0),
            'rejected_input_ids':      rejected['input_ids'].squeeze(0),
            'rejected_attention_mask': rejected['attention_mask'].squeeze(0),
        }


dataset    = PreferenceDataset(PREFERENCE_DATA, tokenizer)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=4, shuffle=True)

print(f"Dataset size: {len(dataset)} preference pairs")
print(f"Batches per epoch: {len(dataloader)}")
print()

# Inspect a single batch to confirm shapes are as expected
batch = next(iter(dataloader))
print("Batch tensor shapes:")
for k, v in batch.items():
    print(f"  {k:<35} {tuple(v.shape)}")

In [ ]:
# Reward model training loop.
#
# The training procedure follows the InstructGPT paper with a few additions:
#   - AdamW optimiser with weight decay for regularisation
#   - Linear warmup then cosine decay for the learning rate
#   - Gradient clipping to prevent the exploding gradient problem,
#     which is especially common in reward model training due to the
#     difference of scores having higher variance than individual scores
#   - Per-epoch tracking of preference accuracy (not just loss)
#     because loss and accuracy can diverge when there is margin saturation
#
# We log both the loss and the preference accuracy at each epoch.
# During a healthy training run, both should improve monotonically.
# If accuracy plateaus while loss continues decreasing, the model is
# likely increasing the margin on already-correct pairs rather than
# learning to rank new pairs correctly (a sign of overfitting).

from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

EPOCHS = 5
LR     = 1e-4

optimiser = AdamW(
    reward_model.parameters(),
    lr=LR,
    weight_decay=0.01,
)

scheduler = OneCycleLR(
    optimiser,
    max_lr=LR,
    steps_per_epoch=len(dataloader),
    epochs=EPOCHS,
    pct_start=0.1,    # 10% of training is warmup
)

train_losses     = []
train_accuracies = []

reward_model.train()

for epoch in range(EPOCHS):
    epoch_loss   = 0.0
    epoch_acc    = 0.0
    n_batches    = 0

    for batch in dataloader:
        # Move data to the target device
        chosen_ids   = batch['chosen_input_ids'].to(device)
        chosen_mask  = batch['chosen_attention_mask'].to(device)
        rejected_ids = batch['rejected_input_ids'].to(device)
        rejected_mask= batch['rejected_attention_mask'].to(device)

        # Forward pass: compute reward for chosen and rejected separately.
        # We run two forward passes rather than concatenating into one batch
        # to keep the implementation clear. In production, concatenate and
        # split after the forward pass for better GPU utilisation.
        r_chosen   = reward_model(chosen_ids,   chosen_mask)
        r_rejected = reward_model(rejected_ids, rejected_mask)

        loss = bradley_terry_loss(r_chosen, r_rejected)
        acc  = preference_accuracy(r_chosen, r_rejected)

        # Backward pass and optimiser step
        optimiser.zero_grad()
        loss.backward()
        # Gradient clipping: essential for reward model stability
        torch.nn.utils.clip_grad_norm_(reward_model.parameters(), max_norm=1.0)
        optimiser.step()
        scheduler.step()

        epoch_loss += loss.item()
        epoch_acc  += acc
        n_batches  += 1

    avg_loss = epoch_loss / n_batches
    avg_acc  = epoch_acc  / n_batches
    train_losses.append(avg_loss)
    train_accuracies.append(avg_acc)

    print(f"Epoch {epoch+1}/{EPOCHS}  loss={avg_loss:.4f}  preference_accuracy={avg_acc:.1%}")

print("\nTraining complete.")

In [ ]:
# Plot training curves for the reward model.
#
# We display loss and preference accuracy on a shared x-axis to show
# whether they move together. Healthy training shows both curves improving.
#
# A common failure mode is loss continuing to fall after accuracy has
# plateaued. This happens when the model has learned the correct ranking
# for all pairs in the training set but is now only increasing the score
# margin (making already-correct differences larger). This provides no
# generalisation benefit and is the reward model equivalent of overfitting.

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

epochs_range = range(1, EPOCHS + 1)

ax = axes[0]
ax.plot(epochs_range, train_losses, 'o-', color='#1f77b4', linewidth=2, markersize=6)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Bradley-Terry Loss', fontsize=11)
ax.set_title('Reward Model Training Loss', fontsize=12)
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(epochs_range, [a * 100 for a in train_accuracies],
        'o-', color='#2ca02c', linewidth=2, markersize=6)
ax.axhline(50, color='gray', linestyle='--', alpha=0.6, label='Random baseline (50%)')
ax.axhline(75, color='orange', linestyle=':', alpha=0.7, label='Good RM threshold (~75%)')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Preference Accuracy (%)', fontsize=11)
ax.set_title('Reward Model Preference Accuracy', fontsize=12)
ax.set_ylim(0, 105)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.suptitle('Reward Model Training Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/08_rm_training.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Reward Model Failure Modes

Even a well-trained reward model will have systematic biases. Understanding these biases is critical for building robust RLHF pipelines.

In [ ]:
# Measure length bias in the trained reward model.
#
# Length bias is the most widely documented reward model failure mode.
# Human annotators tend to rate longer responses higher, all else being equal,
# because length is a surface proxy for thoroughness. The reward model learns
# this correlation from the training data.
#
# We measure this by generating responses at three length levels with
# identical content, then checking whether the reward model ranks longer
# responses higher even when longer does not mean better.
#
# If the reward model has length bias, the RLHF policy will learn to produce
# unnecessarily long responses (verbosity) to maximise reward.

reward_model.eval()

base_prompt = "### Instruction:\nWhat is the learning rate in neural network training?\n\n### Response:\n"

# Three responses with identical core content but different padding
response_variants = [
    {
        'label': 'Concise (30 words)',
        'text': base_prompt + "The learning rate controls how large each parameter update step is during gradient descent. A small rate converges slowly; a large rate may overshoot the minimum.",
    },
    {
        'label': 'Medium (80 words)',
        'text': base_prompt + (
            "The learning rate is a hyperparameter that controls the step size taken during "
            "gradient descent optimisation. At each training step, parameters are updated "
            "by subtracting the gradient multiplied by the learning rate. A value too small "
            "causes slow convergence; a value too large causes the loss to oscillate or "
            "diverge. Common values range from 1e-5 to 1e-3 for transformer fine-tuning. "
            "Scheduling strategies like cosine annealing progressively reduce the rate."
        ),
    },
    {
        'label': 'Verbose (150 words, padded)',
        'text': base_prompt + (
            "The learning rate is an extremely important hyperparameter in neural network "
            "training that controls the step size. Let me explain this thoroughly. During "
            "gradient descent, we compute the gradient of the loss with respect to each "
            "parameter. We then subtract the gradient multiplied by the learning rate from "
            "each parameter. If the learning rate is too small, training is slow and may "
            "get stuck in local minima. If it is too large, the loss will oscillate or "
            "diverge completely. For transformers the typical range is 1e-5 to 1e-3. "
            "Many practitioners use cosine annealing or linear warmup schedules. The "
            "AdamW optimiser has per-parameter adaptive learning rates which reduces "
            "sensitivity to the global learning rate somewhat. These are all important "
            "considerations for successful neural network training."
        ),
    },
]

print("Length bias measurement:")
print(f"  {'Variant':<25} {'Words':>6} {'Reward Score':>14}")
print("  " + "-" * 48)

scores = []
for variant in response_variants:
    enc = tokenizer(
        variant['text'],
        return_tensors='pt',
        truncation=True,
        max_length=256,
        padding='max_length',
    ).to(device)
    with torch.no_grad():
        score = reward_model(enc['input_ids'], enc['attention_mask']).item()
    word_count = len(variant['text'].split())
    scores.append(score)
    print(f"  {variant['label']:<25} {word_count:>6} {score:>14.4f}")

print()
if scores[2] > scores[0]:
    print("Length bias detected: the verbose response scored higher despite no additional")
    print("quality. In RLHF, this would reward the policy for padding responses.")
else:
    print("No strong length bias on this small dataset -- expected with limited training.")
    print("In production reward models trained on large datasets, this bias is well documented.")

In [ ]:
# Reward model calibration analysis.
#
# A well-calibrated reward model should not just rank responses correctly
# but also express appropriate confidence. When it outputs a large score
# difference, the chosen response should win with high probability on
# held-out human judgements.
#
# We simulate a calibration analysis using synthetic confidence scores
# and hypothetical ground-truth human ratings. In a real project you would
# collect these ground truths through a separate evaluation round.
#
# The calibration plot (confidence vs observed accuracy) should ideally
# fall on the diagonal. Points above the diagonal indicate underconfidence;
# points below indicate overconfidence.

# Generate synthetic calibration data representing a reward model's
# score differences and whether the prediction matched human judgment
np.random.seed(42)
n_eval = 200

# Score differences for evaluation pairs
score_diffs = np.random.randn(n_eval) * 1.5

# Probability that chosen is truly better, given the score difference
# A perfectly calibrated model has P(chosen wins) = sigmoid(diff)
true_prob = 1 / (1 + np.exp(-score_diffs))

# Simulate human judgements: stochastic, not perfectly aligned with RM scores
# Add noise to simulate imperfect but real human agreement (~70% inter-annotator)
human_judgment = np.random.binomial(1, true_prob * 0.85 + 0.075)  # slight miscalibration

# Compute predicted probability from score difference
predicted_prob = 1 / (1 + np.exp(-score_diffs))

# Bucket into confidence bins for the calibration plot
n_bins = 10
bin_edges    = np.linspace(0, 1, n_bins + 1)
bin_centers  = (bin_edges[:-1] + bin_edges[1:]) / 2
obs_accuracy = np.zeros(n_bins)
bin_counts   = np.zeros(n_bins)

for i in range(n_bins):
    mask = (predicted_prob >= bin_edges[i]) & (predicted_prob < bin_edges[i+1])
    if mask.sum() > 0:
        obs_accuracy[i] = human_judgment[mask].mean()
        bin_counts[i]   = mask.sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Calibration plot
ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
valid = bin_counts > 0
scatter = ax.scatter(
    bin_centers[valid], obs_accuracy[valid],
    s=bin_counts[valid] * 5,
    c=bin_centers[valid], cmap='RdYlGn', vmin=0, vmax=1,
    zorder=5, edgecolors='gray', linewidths=0.5
)
ax.plot(bin_centers[valid], obs_accuracy[valid], 'gray', alpha=0.4)
plt.colorbar(scatter, ax=ax, label='Confidence')
ax.set_xlabel('Model Confidence (predicted probability)', fontsize=11)
ax.set_ylabel('Observed Accuracy (human judgement)', fontsize=11)
ax.set_title('Reward Model Calibration\n(Bubble size = number of examples in bin)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# Score distribution
ax = axes[1]
ax.hist(score_diffs[human_judgment == 1], bins=25, alpha=0.7,
        color='#2ca02c', label='Chosen truly preferred', density=True)
ax.hist(score_diffs[human_judgment == 0], bins=25, alpha=0.7,
        color='#d62728', label='Rejected truly preferred', density=True)
ax.axvline(0, color='black', linestyle='--', alpha=0.6, label='Decision boundary')
ax.set_xlabel('Score Difference (r_chosen - r_rejected)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Score Distribution by True Human Preference', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.suptitle('Reward Model Calibration Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/08_rm_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

# Expected Calibration Error (ECE): mean absolute deviation from perfect calibration
ece = np.average(np.abs(obs_accuracy[valid] - bin_centers[valid]),
                 weights=bin_counts[valid])
print(f"Expected Calibration Error (ECE): {ece:.4f}")
print("(ECE = 0 means perfectly calibrated; ECE = 0.5 means maximally miscalibrated)")

## 7. Using the HuggingFace RewardTrainer

The `trl` library provides a production-ready `RewardTrainer` that handles padding, tokenisation, and the Bradley-Terry loss automatically.

In [ ]:
# Production reward model training with the trl RewardTrainer.
#
# RewardTrainer expects a dataset with 'chosen' and 'rejected' keys containing
# the full formatted strings (prompt + response). It handles tokenisation
# internally and applies the Bradley-Terry loss automatically.
#
# The most important argument is 'max_length', which controls how many tokens
# are used for each example. Longer sequences capture more context but require
# more memory. The sweet spot depends on the average length of your responses.
#
# RewardModelConfig integrates with LoRA via the peft argument, allowing
# reward model training with significantly reduced memory requirements.
# This is valuable when your base model is 7B or larger.

from datasets import Dataset
from trl import RewardTrainer, RewardConfig
from peft import LoraConfig, TaskType

# Convert our preference data to the format RewardTrainer expects:
# each row has 'chosen' and 'rejected' fields with full formatted text
reward_dataset = Dataset.from_list([
    {
        'chosen':   f"### Instruction:\n{p.prompt}\n\n### Response:\n{p.chosen}",
        'rejected': f"### Instruction:\n{p.prompt}\n\n### Response:\n{p.rejected}",
    }
    for p in PREFERENCE_DATA[:20]  # small subset for demo
])

print(f"Reward training dataset: {len(reward_dataset)} pairs")
print(f"Sample chosen text length: {len(reward_dataset[0]['chosen'])} characters")
print()

# LoRA configuration for memory-efficient reward model training
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['c_attn'],   # GPT-2 attention projection
    task_type=TaskType.SEQ_CLS,  # Classification task type for reward head
    bias='none',
)

# RewardConfig mirrors TrainingArguments but adds reward-specific options
reward_config = RewardConfig(
    output_dir='./reward_model_output',
    num_train_epochs=2,
    per_device_train_batch_size=2,
    learning_rate=1e-4,
    logging_steps=5,
    max_length=256,          # Maximum sequence length for reward model
    report_to='none',
    dataloader_num_workers=0,
)

# Fresh reward model for this demo
from transformers import AutoModelForSequenceClassification

# trl's RewardTrainer expects an AutoModelForSequenceClassification with num_labels=1
reward_model_hf = AutoModelForSequenceClassification.from_pretrained(
    'gpt2',
    num_labels=1,   # single scalar output
)
reward_model_hf.config.pad_token_id = tokenizer.pad_token_id

trainer = RewardTrainer(
    model=reward_model_hf,
    args=reward_config,
    train_dataset=reward_dataset,
    tokenizer=tokenizer,
    peft_config=lora_config,
)

print("Starting training with RewardTrainer...")
trainer.train()
print("Done. The trained reward model is ready to be used as the reward signal in PPO.")

## 8. Engineering Notes

**Reward model training checklist**

| Concern | Recommendation |
|---------|---------------|
| Initialisation | Always start from the SFT checkpoint, not the base model |
| Data size | Minimum 10,000 pairs; 100,000+ for production quality |
| Length bias | Add a length penalty to the final reward: r_adjusted = r - alpha * len(response) |
| Overoptimisation | Use KL penalty during PPO; stop RL when true eval degrades |
| Distribution shift | Continuously collect preferences on outputs of the current policy |
| Calibration | Measure ECE on held-out set; recalibrate with temperature scaling if needed |
| Multiple rewards | Consider separate reward models for helpfulness and safety |

## 9. Exercises

1. **Margin effect**: Train two reward models -- one with `margin=0.0` and one with `margin=0.5` in the Bradley-Terry loss. Compare their score distributions and preference accuracy on a held-out set. When is a margin beneficial?

2. **Length bias measurement**: After training, create 10 pairs of responses with identical content but different lengths (50 vs 150 words). Compute the Spearman rank correlation between length and reward score. If the correlation is positive and significant, the model has length bias.

3. **Data quality sensitivity**: Train reward models on datasets with 5%, 10%, and 20% of labels corrupted (chosen/rejected swapped). Plot preference accuracy vs corruption rate. At what corruption level does the reward model become unreliable?

4. **Ensemble reward models**: Train three reward models with different random seeds and different LoRA ranks. Average their scores at inference time. Does ensembling improve robustness to reward hacking compared to a single model?

5. **Reward model generalisation**: Train on 'explanation' category examples only. Evaluate on 'code' and 'safety' examples. Does the model generalise? What does this tell you about data coverage requirements?

---

## 10. References

- [Ouyang et al. (2022) -- InstructGPT: Training language models to follow instructions](https://arxiv.org/abs/2203.02155)
- [Gao et al. (2023) -- Scaling Laws for Reward Model Overoptimization](https://arxiv.org/abs/2210.10760)
- [Dubois et al. (2023) -- AlpacaFarm: A Simulation Framework for Methods that Learn from Human Feedback](https://arxiv.org/abs/2305.14387)
- [trl RewardTrainer Documentation](https://huggingface.co/docs/trl/reward_trainer)